# Validate TTD Logic — Scenarios Sweep

Runs the **legacy** `validate_ttd_logic.ipynb` aggregation (per-run mean of fault scores → mean/median/stdev across runs) against all 5 synthetic scenario JSONs. Output is saved to `validate_ttd_logic_scenarios_output.json`.

In [1]:
import json, statistics
from pathlib import Path
from pprint import pprint

SCEN_DIR = Path('data/scenarios')
OUT_FILE = Path('validate_ttd_logic_scenarios_output.json')

SCENARIOS = [
    ('scenario1_normal',              'scenario1_normal.json'),
    ('scenario2_null_heavy',          'scenario2_null_heavy.json'),
    ('scenario3_low_variance',        'scenario3_low_variance.json'),
    ('scenario4_sla_breach',          'scenario4_sla_breach.json'),
    ('scenario5_small_sample',        'scenario5_small_sample.json'),
    ('scenario6_category_imbalance',  'scenario6_category_imbalance.json'),
    ('scenario7a_small_n_baseline',   'scenario7a_small_n_baseline.json'),
    ('scenario7b_small_n_blind_median','scenario7b_small_n_blind_median.json'),
]
for name, fn in SCENARIOS:
    print(f"{name:35} exists={(SCEN_DIR/fn).exists()}")

scenario1_normal                    exists=True
scenario2_null_heavy                exists=True
scenario3_low_variance              exists=True
scenario4_sla_breach                exists=True
scenario5_small_sample              exists=True
scenario6_category_imbalance        exists=True
scenario7a_small_n_baseline         exists=True
scenario7b_small_n_blind_median     exists=True


## Legacy normalization + aggregation (copied verbatim from `validate_ttd_logic.ipynb`)

In [2]:
def normalize_speed_sla_aware(mean_seconds, sla_threshold):
    if mean_seconds is None or mean_seconds == 0:
        return 0.0
    ratio = mean_seconds / sla_threshold
    if ratio <= 1.0:
        return 1 - ratio * 0.85
    return max(0, 0.15 - (ratio - 1.0) * 0.3)

def normalize_ttd_with_fault_sla(fault_name, ttd_value, sla_dict):
    if fault_name not in sla_dict or sla_dict[fault_name] is None:
        return None, None
    sla = sla_dict[fault_name]
    if sla in (None, 0):
        return None, sla
    if ttd_value is None or ttd_value == 0:
        return 0.0, sla
    return normalize_speed_sla_aware(ttd_value, sla), sla

def legacy_aggregate(ttd_dict, sla_dict):
    normalized = {}
    category_by_run = {}
    for run_id, cats in ttd_dict.items():
        normalized[run_id] = {}
        category_by_run[run_id] = {}
        for cat, faults in cats.items():
            normalized[run_id][cat] = {}
            fs = []
            for fname, ttd in faults.items():
                score, sla = normalize_ttd_with_fault_sla(fname, ttd, sla_dict)
                normalized[run_id][cat][fname] = {'ttd': ttd, 'sla': sla, 'score': score}
                if score is not None:
                    fs.append(score)
            if fs:
                category_by_run[run_id][cat] = {
                    'mean': sum(fs)/len(fs), 'min': min(fs), 'max': max(fs), 'count': len(fs)
                }
            else:
                category_by_run[run_id][cat] = {'mean': None, 'count': 0}
    overall = {}
    for run_id, cats in category_by_run.items():
        for cat, stats in cats.items():
            if stats['mean'] is not None:
                overall.setdefault(cat, []).append(stats['mean'])
    category_overall = {}
    for cat, scores in overall.items():
        category_overall[cat] = {
            'mean':   round(sum(scores)/len(scores), 3),
            'median': round(statistics.median(scores), 3),
            'stdev':  round(statistics.stdev(scores) if len(scores) > 1 else 0.0, 3),
            'min':    round(min(scores), 3),
            'max':    round(max(scores), 3),
            'count_runs': len(scores),
        }
    return normalized, category_by_run, category_overall

## Run all 5 scenarios

In [3]:
all_results = {}
for name, fn in SCENARIOS:
    data = json.loads((SCEN_DIR / fn).read_text())
    sla  = data['sla']
    runs = data['runs']

    valid_count = sum(1 for r in runs.values() for c in r.values() for v in c.values() if v not in (None, 0))
    null_count  = sum(1 for r in runs.values() for c in r.values() for v in c.values() if v is None)
    zero_count  = sum(1 for r in runs.values() for c in r.values() for v in c.values() if v == 0)

    _, _, category_overall = legacy_aggregate(runs, sla)

    print(f"\n{'='*70}\n{name}  (n_runs={len(runs)}  valid={valid_count}  null={null_count}  zero={zero_count})\n{'='*70}")
    for cat in sorted(category_overall.keys()):
        s = category_overall[cat]
        print(f"  {cat:18}  mean={s['mean']:.3f}  median={s['median']:.3f}  stdev={s['stdev']:.3f}  range={s['min']}–{s['max']}  runs={s['count_runs']}")

    all_results[name] = {
        'file': fn,
        'n_runs': len(runs),
        'n_valid_obs': valid_count,
        'n_null_obs': null_count,
        'n_zero_obs': zero_count,
        'sla': sla,
        'category_overall': category_overall,
    }


scenario1_normal  (n_runs=30  valid=90  null=0  zero=0)
  network_fault       mean=0.677  median=0.680  stdev=0.100  range=0.496–0.823  runs=30
  resource_fault      mean=0.653  median=0.663  stdev=0.058  range=0.55–0.753  runs=30

scenario2_null_heavy  (n_runs=30  valid=30  null=60  zero=0)
  network_fault       mean=0.156  median=0.000  stdev=0.297  range=0.0–0.819  runs=30
  resource_fault      mean=0.208  median=0.229  stdev=0.195  range=0.0–0.683  runs=30

scenario3_low_variance  (n_runs=30  valid=90  null=0  zero=0)
  network_fault       mean=0.663  median=0.664  stdev=0.027  range=0.607–0.721  runs=30
  resource_fault      mean=0.667  median=0.666  stdev=0.020  range=0.634–0.73  runs=30

scenario4_sla_breach  (n_runs=30  valid=81  null=9  zero=0)
  network_fault       mean=0.116  median=0.031  stdev=0.155  range=0.0–0.472  runs=30
  resource_fault      mean=0.135  median=0.083  stdev=0.140  range=0.0–0.493  runs=30

scenario5_small_sample  (n_runs=5  valid=11  null=4  zero=0)
 

## Save consolidated output

In [4]:
OUT_FILE.write_text(json.dumps(all_results, indent=2))
print(f'Wrote {OUT_FILE.resolve()}  ({OUT_FILE.stat().st_size/1024:.2f} KB)')
pprint(all_results, width=110, depth=4)

Wrote C:\Users\meemankgupta\Music\Project\infosys\certifier\metrics_extractor\notebooks\validate_ttd_logic_scenarios_output.json  (6.04 KB)
{'scenario1_normal': {'category_overall': {'network_fault': {'count_runs': 30,
                                                             'max': 0.823,
                                                             'mean': 0.677,
                                                             'median': 0.68,
                                                             'min': 0.496,
                                                             'stdev': 0.1},
                                           'resource_fault': {'count_runs': 30,
                                                              'max': 0.753,
                                                              'mean': 0.653,
                                                              'median': 0.663,
                                                              'min': 0.55,
               